# Project 55 — Local GitHub Repo Reader Agent
## Parse Code with AST, Build Embeddings, Answer Questions

**Stack:** LangChain · Ollama · ChromaDB · ast · Jupyter

In [ ]:
# !pip install -q langchain langchain-ollama langchain-community chromadb

## Step 1 — Create Sample Codebase

In [ ]:
from pathlib import Path
import ast, json

Path("sample_repo").mkdir(exist_ok=True)
Path("sample_repo/models").mkdir(exist_ok=True)
Path("sample_repo/utils").mkdir(exist_ok=True)

(Path("sample_repo/models/user.py")).write_text('''
class User:
    """Represents a user in the system."""
    def __init__(self, name: str, email: str, role: str = "viewer"):
        self.name = name
        self.email = email
        self.role = role

    def is_admin(self) -> bool:
        """Check if user has admin privileges."""
        return self.role == "admin"

    def to_dict(self) -> dict:
        return {"name": self.name, "email": self.email, "role": self.role}
''')

(Path("sample_repo/models/order.py")).write_text('''
from datetime import datetime

class Order:
    """Represents a customer order."""
    def __init__(self, user_id: int, items: list, total: float):
        self.user_id = user_id
        self.items = items
        self.total = total
        self.created_at = datetime.now()
        self.status = "pending"

    def confirm(self):
        """Mark order as confirmed."""
        self.status = "confirmed"

    def cancel(self):
        """Cancel the order if still pending."""
        if self.status == "pending":
            self.status = "cancelled"
            return True
        return False
''')

(Path("sample_repo/utils/validators.py")).write_text('''
import re

def validate_email(email: str) -> bool:
    """Validate email format."""
    pattern = r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}$'
    return bool(re.match(pattern, email))

def validate_password(password: str) -> dict:
    """Check password strength."""
    checks = {
        "min_length": len(password) >= 8,
        "has_upper": any(c.isupper() for c in password),
        "has_digit": any(c.isdigit() for c in password),
    }
    checks["valid"] = all(checks.values())
    return checks
''')

print("Sample repo created with 3 modules")

## Step 2 — AST-Based Code Analysis

In [ ]:
def extract_code_info(filepath):
    """Extract classes, methods, functions, docstrings using AST."""
    source = Path(filepath).read_text(encoding="utf-8")
    try:
        tree = ast.parse(source)
    except SyntaxError:
        return {"error": "syntax error", "file": filepath}
    info = {"file": filepath, "classes": [], "functions": [], "imports": []}
    for node in ast.walk(tree):
        if isinstance(node, ast.ClassDef):
            methods = []
            for item in node.body:
                if isinstance(item, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    methods.append({
                        "name": item.name,
                        "args": [a.arg for a in item.args.args if a.arg != "self"],
                        "doc": ast.get_docstring(item) or "",
                        "line": item.lineno,
                    })
            info["classes"].append({
                "name": node.name,
                "doc": ast.get_docstring(node) or "",
                "methods": methods,
                "line": node.lineno,
            })
        elif isinstance(node, ast.FunctionDef) and not isinstance(
            getattr(node, '_parent', None), ast.ClassDef
        ):
            info["functions"].append({
                "name": node.name,
                "doc": ast.get_docstring(node) or "",
                "args": [a.arg for a in node.args.args],
                "line": node.lineno,
            })
        elif isinstance(node, (ast.Import, ast.ImportFrom)):
            names = [a.name for a in node.names]
            info["imports"].extend(names)
    return info

# Analyze all files
repo_info = []
for py in Path("sample_repo").rglob("*.py"):
    info = extract_code_info(str(py))
    repo_info.append(info)
    classes = [c["name"] for c in info.get("classes", [])]
    funcs = [f["name"] for f in info.get("functions", [])]
    print(f"  {py}: classes={classes}, functions={funcs}")

## Step 3 — Build Code Embeddings

In [ ]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
import shutil

embeddings = OllamaEmbeddings(model="nomic-embed-text")
llm = ChatOllama(model="qwen3:8b", temperature=0.1)

# Create documents from code analysis
docs = []
for info in repo_info:
    fp = info["file"]
    source = Path(fp).read_text(encoding="utf-8")
    # One doc per class/function
    for cls in info.get("classes", []):
        text = f"Class {cls['name']} in {fp}: {cls['doc']}\nMethods: "
        text += ", ".join([m["name"] + "(" + ",".join(m["args"]) + ")" for m in cls["methods"]])
        docs.append(Document(page_content=text, metadata={"file": fp, "type": "class", "name": cls["name"]}))
    for func in info.get("functions", []):
        text = f"Function {func['name']}({','.join(func['args'])}) in {fp}: {func['doc']}"
        docs.append(Document(page_content=text, metadata={"file": fp, "type": "function", "name": func["name"]}))
    # Full file doc
    docs.append(Document(page_content=source[:500], metadata={"file": fp, "type": "source"}))

shutil.rmtree("chroma_repo", ignore_errors=True)
store = Chroma.from_documents(docs, embeddings, persist_directory="chroma_repo")
retriever = store.as_retriever(search_kwargs={"k": 3})
print(f"Indexed {len(docs)} code documents")

## Step 4 — Code Q&A

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert code analyst. Answer questions about the codebase using the context provided."),
    ("human", "Context:\n{context}\n\nQuestion: {question}")
])
qa_chain = qa_prompt | llm | StrOutputParser()

questions = [
    "How do I create a new user?",
    "Can an order be cancelled after confirmation?",
    "How is email validation implemented?",
    "What methods does the User class have?",
]

for q in questions:
    docs_found = retriever.invoke(q)
    context = "\n---\n".join([d.page_content for d in docs_found])
    answer = qa_chain.invoke({"context": context, "question": q})
    print(f"Q: {q}")
    print(f"A: {answer[:250]}")
    print()

## Step 5 — Code Improvement Suggestions

In [ ]:
from pydantic import BaseModel, Field

class CodeReview(BaseModel):
    file: str
    issues: list[str]
    improvements: list[str]
    security_notes: list[str]
    overall_quality: str = Field(description="good, fair, poor")

reviewer = llm.with_structured_output(CodeReview)

for info in repo_info:
    source = Path(info["file"]).read_text("utf-8")
    review = reviewer.invoke(f"Review this code for quality, bugs, security:\n{source}")
    print(f"\n{info['file']} — {review.overall_quality}")
    for issue in review.issues:
        print(f"  ✗ {issue}")
    for imp in review.improvements:
        print(f"  → {imp}")

## What You Learned
- **AST-based code parsing** for structure extraction
- **Code embeddings** in ChromaDB for semantic search
- **Codebase Q&A** with retrieval-augmented generation
- **Automated code review** with structured output